In [1]:
!pip install sacremoses kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 12.6 MB/s eta 0:00:0000:010:01


In [11]:
import pandas as pd

# Load your 92k merged file
df = pd.read_csv('/kaggle/input/datasets/tarak57/v11-pro-c1/V11_PRO_CHECKPOINT_MERGED.csv')

# Drop all the images that got trapped by the Jaccard check
df_clean = df[df['Gate_Decision'] != 'Uncertain']

# Save the clean checkpoint
df_clean.to_csv('V11_PRO_CLEAN_CHECKPOINT.csv', index=False)

print(f"Cleaned! Remaining rows: {len(df_clean)}. Upload this as your new Kaggle Dataset.")

Cleaned! Remaining rows: 44392. Upload this as your new Kaggle Dataset.


In [12]:
import os
import shutil
INPUT_CHECKPOINT = '/kaggle/input/datasets/tarak57/v11-pro-c1/V11_PRO_CHECKPOINT_MERGED.csv'
WORKING_DESTINATION = '/kaggle/working/v11_pro_audit.csv'

if os.path.exists(INPUT_CHECKPOINT) and not os.path.exists(WORKING_DESTINATION):
    print("Detected clean checkpoint in input folder. Copying to working directory...")
    shutil.copy(INPUT_CHECKPOINT, WORKING_DESTINATION)

In [13]:
import os
import glob
import torch
import re
import string
import shutil
import pandas as pd
import numpy as np
import joblib
import warnings
import transformers
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from tqdm import tqdm
from transformers import VisionEncoderDecoderModel, AutoTokenizer, ViTImageProcessor
from transformers import LogitsProcessor, LogitsProcessorList
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV

# Silence Warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

print("==================================================")
print("PRODUCTION DEPLOYMENT: V11-PRO (TRAP-FREE CALIBRATED)")
print("==================================================")

# ---------------------------------------------------------
# 0. RESUME CHECKPOINT HANDLER
# ---------------------------------------------------------
# Update this path to match your newly uploaded clean checkpoint!
INPUT_CHECKPOINT = '/kaggle/input/v11-pro-checkpoint/V11_PRO_CLEAN_CHECKPOINT.csv'
WORKING_DESTINATION = '/kaggle/working/v11_pro_audit.csv'

if os.path.exists(INPUT_CHECKPOINT) and not os.path.exists(WORKING_DESTINATION):
    print("Detected clean checkpoint in input folder. Copying to working directory...")
    shutil.copy(INPUT_CHECKPOINT, WORKING_DESTINATION)

# ---------------------------------------------------------
# 1. PATH CONFIGURATION & PRE-PROCESSING
# ---------------------------------------------------------
IMAGE_DIR = '/kaggle/input/'
DATA_CSV_PATH = None
WEIGHTS_PATH = None

for root, _, files in os.walk(IMAGE_DIR):
    if 'Data_Entry_2017.csv' in files:
        DATA_CSV_PATH = os.path.join(root, 'Data_Entry_2017.csv')
    if 'config.json' in files:
        WEIGHTS_PATH = root

if not DATA_CSV_PATH or not WEIGHTS_PATH:
    raise FileNotFoundError("Could not find Dataset CSV or Model Weights.")

OUTPUT_CSV = '/kaggle/working/v11_pro_audit.csv'
GATE_MODEL_PATH = '/kaggle/working/v11_pro_mlp_gate_calibrated.pkl'

full_df = pd.read_csv(DATA_CSV_PATH)
full_df['Binary_Label'] = (full_df['Finding Labels'] != "No Finding").astype(int)

CSV_HEADERS = [
    "Image_ID", "Ground_Truth", "Is_Real_Abnormal", 
    "Gate_Decision", "Gate_Confidence", "Raw_Confidence", 
    "Final_Report", "SHR_Flag"
]

if os.path.exists(OUTPUT_CSV):
    completed_df = pd.read_csv(OUTPUT_CSV)
    processed_images = set(completed_df['Image_ID'].tolist())
    print(f"Resuming run: {len(processed_images)} valid images already processed.")
else:
    processed_images = set()
    pd.DataFrame(columns=CSV_HEADERS).to_csv(OUTPUT_CSV, index=False)
    print("Starting fresh production run for V11-PRO.")

remaining_df = full_df[~full_df['Image Index'].isin(processed_images)].copy()
print(f"Images to process in this session: {len(remaining_df)}")

print("Indexing dataset file paths...")
image_map = {f: os.path.join(root, f) for root, _, files in os.walk(IMAGE_DIR) for f in files if f.lower().endswith('.png')}

# ---------------------------------------------------------
# 2. CENTRAL MLP GATE TRAINING (CALIBRATED)
# ---------------------------------------------------------
if not os.path.exists(GATE_MODEL_PATH) and len(remaining_df) > 0:
    print("Training the Calibrated V11-PRO MLP Gate...")
    vlm_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    vit_model = VisionEncoderDecoderModel.from_pretrained(WEIGHTS_PATH).get_encoder().to("cuda:0")
    vit_model.eval()
    
    train_pool = pd.concat([
        full_df[full_df['Binary_Label'] == 0].sample(1000, random_state=42),
        full_df[full_df['Binary_Label'] == 1].sample(1000, random_state=42)
    ])
    
    X_train, y_train = [], []
    for _, row in tqdm(train_pool.iterrows(), total=len(train_pool), desc="Caching Features", leave=False):
        img_path = image_map.get(row["Image Index"])
        if not img_path: continue
        
        with Image.open(img_path) as img:
            pv = vlm_processor(images=img.convert('RGB'), return_tensors="pt").pixel_values.to("cuda:0")
            
        with torch.inference_mode(), torch.amp.autocast('cuda'):
            feats = vit_model(pixel_values=pv).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        X_train.append(feats)
        y_train.append(row['Binary_Label'])
        
    base_mlp = MLPClassifier(hidden_layer_sizes=(256, 64), activation='relu', solver='adam', alpha=1e-4, max_iter=300, early_stopping=True, random_state=42)
    calibrated_gate = CalibratedClassifierCV(base_mlp, method='isotonic', cv=5)
    
    calibrated_gate.fit(X_train, y_train)
    joblib.dump(calibrated_gate, GATE_MODEL_PATH)
    print("Calibrated MLP Gate trained and saved.")
    
    del vit_model, vlm_processor, X_train, y_train, calibrated_gate
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# 3. THE THREAD-SAFE INFERENCE WORKER
# ---------------------------------------------------------
def gpu_worker(gpu_id, chunk_df, image_map, weights_path, gate_path, output_part_csv):
    device = torch.device(f"cuda:{gpu_id}")
    
    tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt", token=False)
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    model = VisionEncoderDecoderModel.from_pretrained(weights_path).to(device)
    vit_encoder = model.get_encoder()
    mlp_gate = joblib.load(gate_path)
    
    class FastLogits(LogitsProcessor):
        def __init__(self, mod_dict, vocab_size):
            self.mod_tensor = torch.zeros(vocab_size, device=device)
            for tid, val in mod_dict.items(): self.mod_tensor[tid] += val
        def __call__(self, input_ids, scores):
            scores += self.mod_tensor
            return scores

    def build_processor(is_abnormal):
        mod_dict = {}
        def add_words(words, val):
            for w in words:
                tokens = tokenizer.encode(w, add_special_tokens=False) + tokenizer.encode(" " + w, add_special_tokens=False)
                for t in set(tokens): mod_dict[t] = mod_dict.get(t, 0.0) + val
                
        add_words(["hardware", "pacemaker", "tube", "catheter", "device", "wire", "implant", "clip", "xxxx", "xxx", "xx", "artifact", "lead", "surgical"], -2.0)
        absolute = ["no focal", "no evidence", "no acute", "clear"]
        anatomy = ["normal", "unremarkable", "intact"]
        
        if is_abnormal:
            add_words(absolute, -0.8) 
            add_words(anatomy, -0.6)
        else:
            add_words(absolute, -0.5)
            add_words(anatomy, -0.5)
            
        add_words(["opacity", "effusion", "cardiomegaly", "consolidation", "atelectasis", "infiltrate", "edema", "pneumothorax", "nodule", "mass", "lesion", "shadow", "density", "thickening", "fibrosis", "calcification", "congestion"], 0.55) 
        add_words(["lung", "lobe", "base", "apex", "cardiac", "pleural", "mediastinum", "hilum", "diaphragm", "zone"], 0.2)
        return LogitsProcessorList([FastLogits(mod_dict, tokenizer.vocab_size)])

    abn_logits = build_processor(True)
    norm_logits = build_processor(False)
    
    normal_template = "Findings: The lungs are clear. No focal consolidation, pleural effusion, or pneumothorax is seen. Heart size and mediastinal contour are within normal limits. Visualized osseous structures are intact."
    uncertain_template = "Findings: Visual features are indeterminate. Subtle opacities cannot be excluded. Recommend clinical correlation or follow-up imaging."
    triggers = ['hardware', 'pacemaker', 'tube', 'catheter', 'device', 'lead', 'implant', 'clip', 'wire', 'surgical']

    def clean_text(raw_text):
        text = re.sub(r'(?i)x{2,}', '', raw_text)
        text = re.sub(r'\s+', ' ', text).replace(" .", ".").strip()
        if not text.endswith('.'): text += '.'
        if len(text) < 20: text = "Findings: Image quality or features are insufficient for reliable interpretation."
        return text

    buffer = []
    iterator = zip(chunk_df['Image Index'], chunk_df['Finding Labels'], chunk_df['Binary_Label'])
    
    total_imgs = len(chunk_df)
    update_interval = max(1, int(total_imgs * 0.02)) # Calculate the 2% mark
    
    for i, (img_name, gt_label, is_abn) in enumerate(iterator):
        
        # Clean 2% Progress Print
        if i > 0 and i % update_interval == 0:
            pct = int((i / total_imgs) * 100)
            print(f"[GPU {gpu_id}] Processing: {pct}% ({i}/{total_imgs} images completed)")
            
        img_path = image_map.get(img_name)
        if not img_path: continue
            
        try:
            with Image.open(img_path) as img:
                pv = processor(images=img.convert('RGB'), return_tensors="pt").pixel_values.to(device, non_blocking=True)
                
            with torch.inference_mode(), torch.amp.autocast('cuda'):
                feats = vit_encoder(pixel_values=pv).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            raw_prob = float(mlp_gate.predict_proba([feats])[0][1])
            prob = raw_prob
            
            low_thresh = 0.45
            high_thresh = 0.65
            
            # Adaptive Thresholds
            if prob < 0.30: high_thresh = 0.70
            elif prob > 0.80: low_thresh = 0.40
            
            if prob < low_thresh:
                report, dec = normal_template, "Normal"
            elif prob > high_thresh:
                prefix = "Findings: There is evidence of" if prob > 0.85 else "Findings: The lungs demonstrate"
                dec_ids = tokenizer(prefix, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
                
                # Temperature Annealing
                dynamic_temp = 0.65 if prob > 0.90 else 0.75
                
                # TRAP REMOVED: Single, high-quality generation
                with torch.inference_mode(), torch.amp.autocast('cuda'):
                    gen_ids = model.generate(
                        pixel_values=pv, decoder_input_ids=dec_ids, max_new_tokens=45, 
                        do_sample=True, temperature=dynamic_temp, top_p=0.9, 
                        top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=3, 
                        logits_processor=abn_logits
                    )
                    report = clean_text(tokenizer.decode(gen_ids[0], skip_special_tokens=True))
                    dec = "Abnormal"
            else:
                report, dec = uncertain_template, "Uncertain"
                
        except Exception as e:
            continue
            
        buffer.append({
            "Image_ID": img_name, "Ground_Truth": gt_label, "Is_Real_Abnormal": is_abn == 1,
            "Gate_Decision": dec, "Gate_Confidence": round(prob, 4), 
            "Raw_Confidence": round(raw_prob, 4),
            "Final_Report": report, "SHR_Flag": 1 if any(t in report.lower() for t in triggers) else 0
        })
        
        if len(buffer) >= 50:
            pd.DataFrame(buffer).to_csv(output_part_csv, mode='a', header=False, index=False)
            buffer = []
        if i > 0 and i % 500 == 0:
            torch.cuda.empty_cache()

    if buffer:
        pd.DataFrame(buffer).to_csv(output_part_csv, mode='a', header=False, index=False)

# ---------------------------------------------------------
# 4. EXECUTION CONTROLLER (JUPYTER THREADING)
# ---------------------------------------------------------
if __name__ == '__main__':
    if len(remaining_df) > 0:
        NUM_GPUS = torch.cuda.device_count()
        print(f"\nHardware Scan: Detected {NUM_GPUS} CUDA device(s).")
        
        if NUM_GPUS < 2:
            print("Deploying in Single-GPU Mode...")
            gpu_worker(0, remaining_df, image_map, WEIGHTS_PATH, GATE_MODEL_PATH, OUTPUT_CSV)
            print("V11-PRO FULL DATASET COMPLETE. Saved to v11_pro_audit.csv.")
            
        else:
            print("Deploying in Dual-GPU Threading Mode (Jupyter Safe)...")
            mid_point = len(remaining_df) // 2
            df_part0 = remaining_df.iloc[:mid_point]
            df_part1 = remaining_df.iloc[mid_point:]
            
            part0_csv = '/kaggle/working/part0_v11_pro.csv'
            part1_csv = '/kaggle/working/part1_v11_pro.csv'
            
            if not os.path.exists(part0_csv): pd.DataFrame(columns=CSV_HEADERS).to_csv(part0_csv, index=False)
            if not os.path.exists(part1_csv): pd.DataFrame(columns=CSV_HEADERS).to_csv(part1_csv, index=False)
            
            with ThreadPoolExecutor(max_workers=2) as executor:
                f0 = executor.submit(gpu_worker, 0, df_part0, image_map, WEIGHTS_PATH, GATE_MODEL_PATH, part0_csv)
                f1 = executor.submit(gpu_worker, 1, df_part1, image_map, WEIGHTS_PATH, GATE_MODEL_PATH, part1_csv)
                
                f0.result()
                f1.result()
            
            print("\nBoth GPUs finished! Merging outputs...")
            df_final = pd.concat([
                pd.read_csv(OUTPUT_CSV) if os.path.exists(OUTPUT_CSV) else pd.DataFrame(columns=CSV_HEADERS),
                pd.read_csv(part0_csv),
                pd.read_csv(part1_csv)
            ]).drop_duplicates(subset=['Image_ID'])
            
            df_final.to_csv(OUTPUT_CSV, index=False)
            os.remove(part0_csv)
            os.remove(part1_csv)
            
            print("V11-PRO FULL DATASET COMPLETE. Saved to v11_pro_audit.csv.")
    else:
        print("Dataset already completed!")

PRODUCTION DEPLOYMENT: V11-PRO (TRAP-FREE CALIBRATED)
Resuming run: 92100 valid images already processed.
Images to process in this session: 20020
Indexing dataset file paths...


Training the Calibrated V11-PRO MLP Gate...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/591 [00:00<?, ?it/s]

Calibrated MLP Gate trained and saved.

Hardware Scan: Detected 2 CUDA device(s).
Deploying in Dual-GPU Threading Mode (Jupyter Safe)...


config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/591 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/591 [00:00<?, ?it/s]

[GPU 0] Processing: 1% (200/10010 images completed)
[GPU 1] Processing: 1% (200/10010 images completed)
[GPU 0] Processing: 3% (400/10010 images completed)
[GPU 1] Processing: 3% (400/10010 images completed)
[GPU 0] Processing: 5% (600/10010 images completed)
[GPU 0] Processing: 7% (800/10010 images completed)
[GPU 1] Processing: 5% (600/10010 images completed)
[GPU 0] Processing: 9% (1000/10010 images completed)
[GPU 1] Processing: 7% (800/10010 images completed)
[GPU 0] Processing: 11% (1200/10010 images completed)
[GPU 1] Processing: 9% (1000/10010 images completed)
[GPU 0] Processing: 13% (1400/10010 images completed)
[GPU 1] Processing: 11% (1200/10010 images completed)
[GPU 0] Processing: 15% (1600/10010 images completed)
[GPU 1] Processing: 13% (1400/10010 images completed)
[GPU 0] Processing: 17% (1800/10010 images completed)
[GPU 1] Processing: 15% (1600/10010 images completed)
[GPU 0] Processing: 19% (2000/10010 images completed)
[GPU 1] Processing: 17% (1800/10010 images com

In [14]:
import pandas as pd

# Load the file you just sent me
df = pd.read_csv('v11_pro_audit.csv')

# Re-create the adaptive threshold logic to find the trapped images
def should_be_abnormal(prob):
    low_thresh, high_thresh = 0.45, 0.65
    if prob < 0.30: high_thresh = 0.70
    elif prob > 0.80: low_thresh = 0.40
    return prob > high_thresh

# Drop the images that were trapped as "Uncertain" but should be "Abnormal"
bad_rows = (df['Gate_Confidence'].apply(should_be_abnormal)) & (df['Gate_Decision'] == 'Uncertain')
df_clean = df[~bad_rows]

# Save the Ultimate Checkpoint
df_clean.to_csv('V11_PRO_ULTIMATE_CHECKPOINT.csv', index=False)

print(f"Cleaned! Remaining perfect rows: {len(df_clean)}. Upload this to Kaggle!")

Cleaned! Remaining perfect rows: 82049. Upload this to Kaggle!
